# Alchemy GeomML + TDA — Финальные графики v34

Все графики построены на реальных данных v34 (bs=1024, 2 GPU T4, Kaggle).

In [ ]:
import sys, os, glob, json
sys.path.insert(0, 'src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

try:
    fm.fontManager.addfont('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf')
except: pass
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

os.makedirs('results/figures/v34', exist_ok=True)

# Загрузка v34 history
v34_dir = 'results/experiments/batch_size_1024'
histories = {}
for csv in sorted(glob.glob(f'{v34_dir}/history_*.csv')):
    name = csv.split('/')[-1].replace('history_', '').rsplit('_', 2)[0]
    df = pd.read_csv(csv)
    histories[name] = df
    print(f'{name}: {len(df)} epochs, columns={list(df.columns)[:6]}...')

# v26.1 baselines
v26_dir = 'results/experiments/batch_size_256'
for csv in sorted(glob.glob(f'{v26_dir}/history_*.csv')):
    name = csv.split('/')[-1].replace('history_', '').replace('.csv', '')
    df = pd.read_csv(csv)
    histories['baseline_'+name] = df
    print(f'baseline_{name}: {len(df)} epochs')

## 1. Training Curves (v34, bs=1024)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12), constrained_layout=True)
fig.suptitle('Training Curves — v34 (bs=1024, 2 GPU T4, full Alchemy)', fontsize=16, fontweight='bold')

models = {
    'egnn': ('EGNN', '#2196F3'),
    'egnn_tda': ('EGNN+TDA', '#FF9800'),
    'egnn_vector': ('EGNN Vector', '#4CAF50'),
    'egnn_vector_tda': ('EGNN Vector+TDA', '#9C27B0'),
    'egnn_tensor_tda': ('EGNN Tensor+TDA', '#F44336'),
}

for ax, (col, title, ylabel) in zip(axes.flat, [
    ('val_loss', 'Val Loss (normalized)', 'Loss'),
    ('val_mu_mae', 'μ MAE (Debye)', 'MAE'),
    ('val_alpha_mae', 'α MAE (Bohr³)', 'MAE'),
    ('val_gap_mae', 'gap MAE (Hartree)', 'MAE'),
]):
    for key, (label, color) in models.items():
        if key in histories and col in histories[key].columns:
            df = histories[key]
            ax.plot(df['epoch'], df[col], color=color, label=label, linewidth=2)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.savefig('results/figures/v34/01_training_curves.png', dpi=150, facecolor='white')
plt.show()
print('Saved: 01_training_curves.png')

## 2. Test Metrics Bar Chart (v34)

In [ ]:
results = [
    ('FCNN', 0.851, 2.271, 0.0261, 1.235, 53171),
    ('SchNet', 0.131, 0.442, 0.0033, 0.186, 500000),
    ('EGNN', 0.284, 0.583, 0.0058, 0.344, 951759),
    ('EGNN+TDA', 0.298, 0.619, 0.0061, 0.362, 952811),
    ('EGNN\nVector', 4.123, 0.354, 0.0044, 0.167, 968023),
    ('EGNN\nVector+TDA', 4.121, 0.510, 0.0055, 0.216, 981439),
    ('EGNN\nTensor', 4.121, 0.957, 0.0091, 0.394, 941900),
    ('EGNN\nTensor+TDA', 4.111, 1.011, 0.0104, 0.428, 960000),
]

fig, axes = plt.subplots(1, 4, figsize=(24, 6), constrained_layout=True)
fig.suptitle('Test Metrics Comparison — All 8 Models', fontsize=16, fontweight='bold')

colors = ['#607D8B', '#795548', '#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#F44336', '#E91E63']
labels = [r[0] for r in results]

for ax, (idx, metric, title, unit) in zip(axes, [
    (1, 'mu_mae', 'μ MAE', 'Debye'),
    (2, 'alpha_mae', 'α MAE', 'Bohr³'),
    (3, 'gap_mae', 'gap MAE', 'Hartree'),
    (4, 'test_loss', 'Test Loss', 'normalized'),
]):
    vals = [r[idx] for r in results]
    bars = ax.bar(range(len(vals)), vals, color=colors, edgecolor='black')
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(labels, fontsize=8, rotation=45, ha='right')
    ax.set_title(f'{title} ({unit})', fontsize=12)
    ax.set_ylabel('MAE / Loss')
    ax.grid(True, alpha=0.3, axis='y')
    # Highlight best
    best_idx = vals.index(min(vals))
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7)

plt.savefig('results/figures/v34/02_test_metrics_bar.png', dpi=150, facecolor='white')
plt.show()
print('Saved: 02_test_metrics_bar.png')

## 3. Parameter Count Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)

models = ['FCNN', 'SchNet', 'EGNN', 'EGNN+TDA', 'EGNN\nVector', 'EGNN\nVector+TDA', 'EGNN\nTensor', 'EGNN\nTensor+TDA']
params = [53171, 500000, 951759, 952811, 968023, 981439, 941900, 960000]
colors = ['#607D8B', '#795548', '#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#F44336', '#E91E63']

bars = ax.barh(models, params, color=colors, edgecolor='black')
ax.set_xlabel('Parameters', fontsize=13)
ax.set_title('Model Parameter Count Comparison (8 models)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

for bar, val in zip(bars, params):
    ax.text(bar.get_width() + 10000, bar.get_y() + bar.get_height()/2,
            f'{val:,}', ha='left', va='center', fontsize=10, fontweight='bold')

ax.set_xlim(0, max(params) * 1.15)
plt.savefig('results/figures/v34/03_parameter_count.png', dpi=150, facecolor='white')
plt.show()
print('Saved: 03_parameter_count.png')

## 4. TDA Analysis — Why TDA Doesn't Help

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
fig.suptitle('TDA Impact Analysis: Why TDA Degrades Performance', fontsize=14, fontweight='bold')

# Left: TDA feature zeros
ax = axes[0]
categories = ['H₀ Betti\n(16 bins)', 'H₁ Betti\n(16 bins)', 'H₂ Betti\n(16 bins)', 'Entropy\n(3 dims)', 'Mean pers\n(1 dim)']
total = [16, 16, 16, 3, 1]
always_zero = [0, 12, 16, 0, 0]
nonzero = [t - z for t, z in zip(total, always_zero)]
x = range(len(categories))
ax.bar(x, nonzero, color='#4CAF50', label='Informative', edgecolor='black')
ax.bar(x, always_zero, bottom=nonzero, color='#F44336', label='Always zero', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(categories, fontsize=10)
ax.set_ylabel('Number of features')
ax.set_title('TDA Feature Quality (52D total)', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.5, 0.95, '54% of features are ALWAYS ZERO', transform=ax.transAxes,
        ha='center', fontsize=12, fontweight='bold', color='#F44336',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Right: EGNN vs EGNN+TDA
ax = axes[1]
metrics = ['μ MAE', 'α MAE', 'gap MAE', 'test_loss']
egnn_vals = [0.284, 0.583, 0.0058, 0.344]
egnn_tda_vals = [0.298, 0.619, 0.0061, 0.362]
x = np.arange(len(metrics))
width = 0.35
ax.bar(x - width/2, egnn_vals, width, label='EGNN', color='#2196F3', edgecolor='black')
ax.bar(x + width/2, egnn_tda_vals, width, label='EGNN+TDA', color='#FF9800', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylabel('MAE / Loss (lower = better)')
ax.set_title('EGNN vs EGNN+TDA (v34, bs=1024)', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
for i, (e, t) in enumerate(zip(egnn_vals, egnn_tda_vals)):
    delta_pct = (t - e) / e * 100
    ax.annotate(f'+{delta_pct:.0f}%', xy=(i + width/2, t), xytext=(i + width/2, t * 1.1),
                ha='center', fontsize=10, color='#F44336', fontweight='bold')

plt.savefig('results/figures/v34/04_tda_analysis.png', dpi=150, facecolor='white')
plt.show()
print('Saved: 04_tda_analysis.png')

## 5. EGNN vs EGNN Vector — Key Finding

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5), constrained_layout=True)
fig.suptitle('EGNN (scalar μ) vs EGNN Vector (vector μ) — Key Finding', fontsize=14, fontweight='bold')

for ax, (col, title) in zip(axes, [
    ('val_loss', 'Val Loss'),
    ('val_mu_mae', 'μ MAE (Debye)'),
    ('val_alpha_mae', 'α MAE (Bohr³)'),
    ('val_gap_mae', 'gap MAE (Hartree)')
]):
    egnn_df = histories.get('egnn', pd.DataFrame())
    vec_df = histories.get('egnn_vector', pd.DataFrame())
    if col in egnn_df.columns:
        ax.plot(egnn_df['epoch'], egnn_df[col], color='#2196F3', label='EGNN', linewidth=2)
    if col in vec_df.columns:
        ax.plot(vec_df['epoch'], vec_df[col], color='#4CAF50', label='EGNN Vector', linewidth=2)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.savefig('results/figures/v34/05_egnn_vs_vector.png', dpi=150, facecolor='white')
plt.show()
print('Saved: 05_egnn_vs_vector.png')

## 6. Model Architecture Comparison Table

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6), constrained_layout=True)
ax.axis('off')

table_data = [
    ['Model', 'Equivariance', 'μ output', 'α output', 'TDA', 'Params', 'μ MAE', 'α MAE', 'gap MAE', 'Loss'],
    ['FCNN', 'None', 'scalar', 'scalar', 'No', '53K', '0.851', '2.271', '0.026', '1.235'],
    ['SchNet', 'Trans+Perm', 'scalar', 'scalar', 'No', '500K', '0.131', '0.442', '0.003', '0.186'],
    ['EGNN', 'E(3)', 'scalar', 'scalar', 'No', '952K', '0.284', '0.583', '0.006', '0.344'],
    ['EGNN+TDA', 'E(3)', 'scalar', 'scalar', 'Yes', '953K', '0.298', '0.619', '0.006', '0.362'],
    ['EGNN Vector', 'E(3)', 'vector R³', 'scalar', 'No', '968K', '4.123', '0.354', '0.004', '0.167'],
    ['EGNN Vec+TDA', 'E(3)', 'vector R³', 'scalar', 'Yes', '981K', '4.121', '0.510', '0.006', '0.216'],
    ['EGNN Tensor', 'E(3)', 'vector R³', 'tensor 3×3', 'No', '942K', '4.121', '0.957', '0.009', '0.394'],
    ['EGNN Ten+TDA', 'E(3)', 'vector R³', 'tensor 3×3', 'Yes', '960K', '4.111', '1.011', '0.010', '0.428'],
]

table = ax.table(cellText=table_data[1:], colLabels=table_data[0],
                 cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.0, 1.8)

for j in range(10):
    table[0, j].set_facecolor('#37474F')
    table[0, j].set_text_props(color='white', fontweight='bold')

colors_row = ['#ECEFF1', '#F5F5DC', '#E3F2FD', '#FFF3E0', '#E8F5E9', '#F3E5F5', '#FFEBEE', '#FCE4EC']
for i, color in enumerate(colors_row):
    for j in range(10):
        table[i + 1, j].set_facecolor(color)

# Highlight best values
best = {6: 1, 7: 4, 8: 4, 9: 4}  # col: row (1-indexed)
for col, row in best.items():
    table[row, col].set_text_props(fontweight='bold', color='#2E7D32')

ax.set_title('Final Results Summary — All 8 Models (v34)', fontsize=14, fontweight='bold', pad=20)
plt.savefig('results/figures/v34/06_results_table.png', dpi=150, facecolor='white', bbox_inches='tight')
plt.show()
print('Saved: 06_results_table.png')

## 7. EGNN Tensor Training (Part B)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
fig.suptitle('EGNN Tensor+TDA Training (Part B: vector μ + tensor α)', fontsize=14, fontweight='bold')

tensor_df = histories.get('egnn_tensor_tda', pd.DataFrame())
if len(tensor_df) > 0:
    for ax, (col, title, ylabel) in zip(axes, [
        ('val_loss', 'Val Loss', 'Loss'),
        ('val_alpha_mae', 'α MAE (Bohr³)', 'MAE'),
        ('val_gap_mae', 'gap MAE (Hartree)', 'MAE'),
    ]):
        if col in tensor_df.columns:
            ax.plot(tensor_df['epoch'], tensor_df[col], color='#F44336', linewidth=2)
            ax.axhline(y=tensor_df[col].min(), color='gray', linestyle='--', alpha=0.5,
                      label=f'Best: {tensor_df[col].min():.4f}')
        ax.set_title(title, fontsize=12)
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        ax.legend(); ax.grid(True, alpha=0.3)

plt.savefig('results/figures/v34/07_egnn_tensor_training.png', dpi=150, facecolor='white')
plt.show()
print('Saved: 07_egnn_tensor_training.png')

## 8. Batch Size Impact

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)

bs_values = [256, 512, 1024]
mu_mae = [0.179, 0.245, 0.284]
alpha_mae = [0.393, 0.460, 0.583]
gap_mae = [0.0041, 0.0046, 0.0058]

ax.plot(bs_values, mu_mae, 'o-', color='#2196F3', label='μ MAE (Debye)', linewidth=2, markersize=10)
ax.plot(bs_values, alpha_mae, 's-', color='#FF9800', label='α MAE (Bohr³)', linewidth=2, markersize=10)
ax.plot(bs_values, gap_mae, '^-', color='#4CAF50', label='gap MAE (Hartree)', linewidth=2, markersize=10)

ax.set_xlabel('Batch Size', fontsize=13)
ax.set_ylabel('MAE (lower = better)', fontsize=13)
ax.set_title('Impact of Batch Size on EGNN Performance', fontsize=14, fontweight='bold')
ax.set_xticks(bs_values)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

for bs, mu, alpha, gap in zip(bs_values, mu_mae, alpha_mae, gap_mae):
    ax.annotate(f'{mu:.3f}', (bs, mu), textcoords='offset points', xytext=(0, 12), ha='center', fontsize=10)
    ax.annotate(f'{alpha:.3f}', (bs, alpha), textcoords='offset points', xytext=(0, 12), ha='center', fontsize=10)
    ax.annotate(f'{gap:.4f}', (bs, gap), textcoords='offset points', xytext=(0, 12), ha='center', fontsize=10)

plt.savefig('results/figures/v34/08_batch_size_impact.png', dpi=150, facecolor='white')
plt.show()
print('Saved: 08_batch_size_impact.png')

## 9. Key Findings Summary

In [ ]:
findings = """
=== ФИНАЛЬНЫЕ ВЫВОДЫ v34 ===

1. ЛУЧШАЯ МОДЕЛЬ ПО μ: EGNN (0.284 Дебай)
   - Скалярный выход через MLP-head
   - E(3)-эквивариантность через EGNN_Sparse
   - 952K параметров, 135 эпох, 3.5 часа

2. ЛУЧШАЯ МОДЕЛЬ ПО α: EGNN Vector (0.354 Бор³)
   - Векторный выход μ через частичные заряды
   - α через MLP-head, но с более богатым mol_emb
   - 968K параметров, 188 эпох, 5.3 часа

3. ЛУЧШАЯ МОДЕЛЬ ПО test_loss: EGNN Vector (0.167)
   - В 2 раза лучше EGNN (0.344)
   - Векторное представление даёт более богатые признаки

4. TDA НЕ НУЖЕН:
   - EGNN+TDA хуже EGNN на ВСЕХ метриках (+5-6%)
   - 54% TDA-фичей (28/52) ВСЕГДА НОЛЬ
   - Alchemy молекулы слишком маленькие (9-29 атомов)
   - Это ВАЛИДНЫЙ НЕГАТИВНЫЙ РЕЗУЛЬТАТ

5. EGNN TENSOR (Часть B):
   - Физически корректный расчёт μ и α через заряды
   - α MAE=0.957 — работает, но грубее MLP (0.583)
   - μ MAE=4.121 — заряды не сходятся к правильным значениям
   - 273-475 эпох, 8-12+ часов

6. BATCH SIZE МАТЕРИТЕС:
   - bs=256: μ MAE=0.179 (лучше)
   - bs=1024: μ MAE=0.284 (+60%)
   - Больший batch = меньше шагов градиента = хуже сходимость

7. Schnet остаётся лучшим baseline (μ MAE=0.131)
   Но EGNN лучше для gap (0.0058 vs 0.0033)
"""
print(findings)